In [ ]:
import os
import shutil
import sys
import torch
import torch.nn as nn
from torchvision.utils import save_image
from torch.utils.data import DataLoader, Subset

# ==========================================
# 1. CLEAN ENVIRONMENT SETUP
# ==========================================
os.chdir('/kaggle/working')
if os.path.exists('Patch-DM'): shutil.rmtree('Patch-DM')

print("📥 Cloning fresh repository...")
!git clone https://github.com/mlpc-ucsd/Patch-DM.git
os.chdir('Patch-DM')
!pip install -q pytorch-lightning==1.9.5 lmdb==1.2.1 lpips pytorch-fid einops
!pip install -q git+https://github.com/openai/CLIP.git

# Clear module cache
for key in list(sys.modules.keys()):
    if any(p in key for p in ['utils', 'model', 'experiment', 'templates', 'config', 'diffusion']):
        del sys.modules[key]

# ==========================================
# 2. SOURCE FIX (Safe Replacement)
# ==========================================
with open('experiment.py', 'r') as f: exp = f.read()
with open('experiment.py', 'w') as f: f.write(exp.replace('from numpy.lib.function_base import flip', 'from numpy import flip'))

# ==========================================
# 3. FIX SEMANTIC PATH
# ==========================================
INPUT_SEMANTIC = "/kaggle/input/datasets/vanditagupta2609/lmdb-input-2/lmdb-input/semantic.pt"
FIXED_SEMANTIC_PATH = '/kaggle/working/semantic_fixed.pt'
if os.path.exists(INPUT_SEMANTIC):
    data = torch.load(INPUT_SEMANTIC, map_location='cpu')
    wrapped = {'model_state_dict': {'semantic_enc.weight': data[list(data.keys())[0]]}} if isinstance(data, dict) and 'model_state_dict' not in data else data
    torch.save(wrapped, FIXED_SEMANTIC_PATH)

# ==========================================
# 4. LOAD BASELINE MODEL & LIBS
# ==========================================
from utils.dataset import FFHQlmdb
from templates import train_autoenc
from experiment import LitModel
from utils.choices import GenerativeType, ModelMeanType, ModelVarType
from diffusion.base import GaussianDiffusionBeatGans

# ---> PATH TO YOUR BASELINE WEIGHTS <---
CKPT_PATH = "/kaggle/input/datasets/vanditagupta2609/baseline-2/Patch-DM/checkpoints/patchdm_baseline/last.ckpt"

print("🧠 Restoring Baseline weights...")
conf = train_autoenc()
conf.batch_size = 4
conf.patch_size = 64
conf.semantic_path = FIXED_SEMANTIC_PATH

model = LitModel.load_from_checkpoint(CKPT_PATH, conf=conf, strict=False).cuda()
model.eval()

# ==========================================
# 5. THE MONKEY PATCH (Fixes Identity Crisis)
# ==========================================
# We wrap the original method and force the correct Enum identities 
# right before the math logic executes. This kills the NotImplementedError.

original_p_mean_variance = GaussianDiffusionBeatGans.p_mean_variance

def patched_p_mean_variance(self, model, x, t, **kwargs):
    # Force the identities to the session's current Enum types
    self.model_mean_type = ModelMeanType.eps
    self.model_var_type = ModelVarType.fixed_small
    return original_p_mean_variance(self, model, x, t, **kwargs)

# Inject the patch into the class definition
GaussianDiffusionBeatGans.p_mean_variance = patched_p_mean_variance
print("🛠️ Monkey Patch applied: Model will now ignore Enum identity errors.")

# ==========================================
# 6. PREPARE 100 TARGET IMAGES
# ==========================================
print("\n1️⃣ Extracting 100 Real Target Images...")
os.makedirs('/kaggle/working/real_images', exist_ok=True)
dataset = FFHQlmdb(path="/kaggle/input/datasets/vanditagupta2609/lmdb-input-2/lmdb-input/ffhq256.lmdb", image_size=256)
dataloader = DataLoader(Subset(dataset, range(100)), batch_size=4, shuffle=False)

for i in range(100):
    save_image((dataset[i]['img'] + 1) / 2, f'/kaggle/working/real_images/{i}.png')

# ==========================================
# 7. GENERATION (DIRECT LOOP)
# ==========================================
print("\n2️⃣ Generating 100 Baseline Reconstructions...")
os.makedirs('/kaggle/working/fake_images_baseline', exist_ok=True)

# Update config and create sampler
model.conf.beatgans_gen_type = GenerativeType.ddpm
sampler = model.conf._make_diffusion_conf(model.conf.T_eval).make_sampler()
global_idx = 0

with torch.no_grad():
    for batch in dataloader:
        real_imgs = batch['img'].cuda()
        b_size = real_imgs.shape[0]
        
        batch_indices = torch.arange(global_idx, global_idx + b_size, dtype=torch.long, device='cuda')
        cond = model.ema_model.encoder(batch_indices)
        
        pos_y, pos_x = torch.meshgrid(torch.arange(5), torch.arange(5), indexing='ij')
        pos = torch.stack([pos_y, pos_x], dim=-1).reshape(-1, 2).cuda().repeat(b_size, 1)
        
        # Pass EVERYTHING the internal loops need
        model_kwargs = {'cond': cond, 'imgs': real_imgs, 'x_start': real_imgs}
        
        # This will now use our Patched logic automatically
        gen_imgs = sampler.p_sample_loop(
            model=model.ema_model,
            shape=real_imgs.shape,
            noise=torch.randn_like(real_imgs),
            all_pos=[pos],
            model_kwargs=model_kwargs,
            patch_size=64
        )
        
        gen_imgs = (gen_imgs + 1) / 2
        for j in range(b_size):
            if global_idx + j < 100:
                save_image(gen_imgs[j], f'/kaggle/working/fake_images_baseline/{global_idx + j}.png')
            
        global_idx += b_size
        print(f"🚀 Progress: {global_idx}/100 Baseline images saved.")

# ==========================================
# 8. CALCULATE FID & ZIP
# ==========================================
print("\n📊 BASELINE FID SCORE:")
!python -m pytorch_fid /kaggle/working/real_images /kaggle/working/fake_images_baseline

print("\n📦 Finalizing results...")
os.chdir('/kaggle/working')
!zip -q -r baseline_results.zip fake_images_baseline/ real_images/
print("✅ Done! Download 'baseline_results.zip'.")